In [8]:

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report
data=pd.read_csv('/content/drive/MyDrive/Listen_work/allComp_data.csv')
X = data.drop('comp_id', axis=1)
y = data['comp_id']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=101,stratify=y)

xgb_clf = xgb.XGBClassifier(
    objective='multi:softmax',
    eval_metric='mlogloss',
    random_state=101,
    device='cuda',
    n_jobs=1,
)

param_grid = {
    'n_estimators': [100, 200, 500],
    'learning_rate': [0.01, 0.1],
    'reg_alpha': [0.0, 0.1, 0.2],
    'reg_lambda': [0.1, 0.2, 0.3],
    'subsample': [0.5, 0.7, 0.8],
    'gamma': [0.1, 0.2, 0.5, 1],
}

search = RandomizedSearchCV(
    estimator=xgb_clf,
    param_distributions=param_grid,
    n_iter=60,
    scoring='roc_auc_ovr',
    cv=4,
    n_jobs=1,
    random_state=101,
    verbose=2,
    refit=True,
)

search.fit(X_train, y_train)


print("Best params:", search.best_params_)
print("Best CV score:", search.best_score_)

# Evaluate on test set
best_model = search.best_estimator_
y_pred = best_model.predict(X_test)
print("\nTest accuracy:", accuracy_score(y_test, y_pred))
print("\n", classification_report(y_test, y_pred))

Fitting 4 folds for each of 60 candidates, totalling 240 fits


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [12:13:15] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


[CV] END gamma=0.1, learning_rate=0.1, n_estimators=200, reg_alpha=0.0, reg_lambda=0.2, subsample=0.5; total time=   2.6s
[CV] END gamma=0.1, learning_rate=0.1, n_estimators=200, reg_alpha=0.0, reg_lambda=0.2, subsample=0.5; total time=   2.0s
[CV] END gamma=0.1, learning_rate=0.1, n_estimators=200, reg_alpha=0.0, reg_lambda=0.2, subsample=0.5; total time=   1.9s
[CV] END gamma=0.1, learning_rate=0.1, n_estimators=200, reg_alpha=0.0, reg_lambda=0.2, subsample=0.5; total time=   2.0s
[CV] END gamma=0.2, learning_rate=0.1, n_estimators=200, reg_alpha=0.2, reg_lambda=0.2, subsample=0.5; total time=   2.0s
[CV] END gamma=0.2, learning_rate=0.1, n_estimators=200, reg_alpha=0.2, reg_lambda=0.2, subsample=0.5; total time=   2.2s
[CV] END gamma=0.2, learning_rate=0.1, n_estimators=200, reg_alpha=0.2, reg_lambda=0.2, subsample=0.5; total time=   2.2s
[CV] END gamma=0.2, learning_rate=0.1, n_estimators=200, reg_alpha=0.2, reg_lambda=0.2, subsample=0.5; total time=   2.0s
[CV] END gamma=0.5, lear

In [6]:
print(xgb.build_info())

{'BUILTIN_PREFETCH_PRESENT': True, 'CUDA_VERSION': [12, 9], 'DEBUG': False, 'GCC_VERSION': [10, 3, 1], 'GLIBC_VERSION': [2, 28], 'MM_PREFETCH_PRESENT': True, 'NCCL_VERSION': [2, 29, 2], 'THRUST_VERSION': [2, 8, 2], 'USE_CUDA': True, 'USE_DLOPEN_NCCL': True, 'USE_FEDERATED': True, 'USE_NCCL': True, 'USE_NVCOMP': False, 'USE_OPENMP': True, 'USE_RMM': False, 'libxgboost': '/usr/local/lib/python3.12/dist-packages/xgboost/lib/libxgboost.so'}


In [7]:
!nvidia-smi


Sun May 17 12:10:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   62C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [9]:
from joblib import dump, load
dump(best_model, 'base_comp_model.joblib')

['base_comp_model.joblib']